# 05B Skill：先知道有，再按需读

有了 `read_file`，助教仍可能读错文件、漏掉条件或把猜测当结论。Agent Skill 是可按需加载的目录：`SKILL.md` 说明何时使用、如何处理任务，必要时再读取参考文件。

本课采用 Agent Skills 的 `SKILL.md` 格式。加载器是课程自编的最小实现，不代表任何产品都采用同一选取算法或权限规则。

**贯穿案例**

同一源码导读任务需要一个可复用的 Agent Skill，但不应把整个 Skill 目录一直塞进请求。`SkillLibrary` 先只提供名称和描述，选中 `repo-guide` 后加载 `SKILL.md` 正文，需要核对分支时再读取参考清单；审计记录同时保留未激活读取和越界读取两类拒绝。


## 学习路线与配套课件

实践 5B：检查 Agent Skill 并输出加载审计（`L05-P02`）。

建议先完成本节概念正课，再进入本实践小节。这个 Notebook 可用独立新内核从头运行。先预测，再执行代码、修改一个条件并解释结果。

对应课件稳定编号：L05.05-S01、L05.05-S02。页数调整不改变这些编号。

按“问题—代码—观察—练习—可复用结果”的顺序学习。先完成练习，再展开参考分析。回放、保存的真实记录和新的在线请求都会明确标注；在线请求默认关闭。

In [1]:
from pathlib import Path
import sys, json, asyncio, importlib.metadata, hashlib, os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "05 MCP 与 Skills 能力扩展"
sys.path.insert(0, str(CHAPTER / "代码"))
sys.path.insert(0, str(ROOT / "02 从大模型到 Agent 组成与最小循环/代码"))
print("MCP SDK:", importlib.metadata.version("mcp"))
print("课堂协议: 2026-07-28；本地 stdio；生成服务仍为 DeepSeek。")
from skill_lab import SkillLibrary
library = SkillLibrary(CHAPTER / "代码/skills")


MCP SDK: 2.1.1
课堂协议: 2026-07-28；本地 stdio；生成服务仍为 DeepSeek。


## 1. 一个 Skill 的三个层次

`代码/skills/repo-guide/SKILL.md` 的 YAML 文件头包含 `name` 和 `description`：前者标识 Skill，后者描述使用场景。正文给出导读步骤。`references/conditions.md` 是解释条件分支时才需要的细节。

规范允许可选脚本和资产，并非每个 Skill 都必须有这些目录。`allowed-tools` 是可选、实验性的元信息，是否支持和怎样执行取决于宿主；写进 Markdown 不会自动授予权限。


In [2]:
skill_path = CHAPTER / "代码/skills/repo-guide/SKILL.md"
print(skill_path.read_text(encoding='utf-8'))
print("这个 Skill 目录包含:", [str(p.relative_to(skill_path.parent))
    for p in skill_path.parent.rglob("*") if p.is_file()])


---
name: repo-guide
description: 带初学者理解小型代码仓库的入口、调用关系和条件分支。用于仓库导读、寻找配置入口、解释源码；每项结论回到实际读取的文件行号。
---
# 仓库导读

先用资料工具查看可读文件，再围绕用户问题读取入口及被调用函数。资料可能不完整；未读到实现时说明缺口，不根据文件名猜测行为。

解释配置、路径、默认值或异常分支时，按需读取 [条件核验清单](references/conditions.md)。该清单提供核验方法，不预先提供任何仓库的答案。

回答先给一条简短调用关系，再把关键条件讲清楚，引用实际文件和行号。给初学者留一个可以回到源码验证的问题。只回答用户所问范围，不为了使用 Skill 扩大任务。

Skill 正文中的步骤不授予运行程序或修改文件的权限。工具可执行范围由运行时和服务端控制。

这个 Skill 目录包含: ['SKILL.md', 'references\\conditions.md']


## 2. 发现阶段只给模型名称和描述

先列目录并解析 YAML 文件头，可以告诉模型有哪些 Skills 可选。模型此时还没有收到正文和参考文件，减少无关信息进入请求。

这里没有把字符数当 Token 数；“渐进加载”是选择何时提供内容，不是模型自动获得磁盘访问能力。


In [3]:
print(json.dumps(library.catalog(), ensure_ascii=False, indent=2))
print("已激活:", library.active)
print("事件:", library.events)
assert not library.active


[
  {
    "name": "repo-guide",
    "description": "带初学者理解小型代码仓库的入口、调用关系和条件分支。用于仓库导读、寻找配置入口、解释源码；每项结论回到实际读取的文件行号。"
  }
]
已激活: set()
事件: [{'stage': 'discover', 'names': ['repo-guide']}]


## 3. 谁来做加载

下面是可直接运行的课堂加载器。先看 `catalog → activate → read_resource` 三个公开方法，再看初始化时的文件头检查。格式校验只是前置条件，不评估 Skill 对任务是否有效。

为方便解释，课堂实现只接收 `name` / `description` 两字段，并对可读参考文件使用清单。完整规范还有其他合法字段；它们不被本课解析器支持，不等于它们违反规范。


In [4]:
import re, yaml
class SkillLibrary:
    def __init__(self, root):
        self.root = Path(root).resolve()
        self.entries, self.active, self.events = {}, set(), []
        for path in sorted(self.root.glob('*/SKILL.md')):
            if path.is_symlink() or path.parent.is_symlink():
                raise ValueError('课堂包不接受符号链接')
            if path.stat().st_size > 16000:
                raise ValueError('课堂 SKILL.md 上限为 16000 字节')
            # 仅读取文件头；发现阶段不把正文交给模型。
            with path.open(encoding='utf-8') as f:
                if f.readline().strip() != '---': raise ValueError('缺少 YAML 文件头')
                header = []
                for line in f:
                    if line.strip() == '---': break
                    header.append(line)
                else: raise ValueError('文件头未关闭')
            info = yaml.safe_load(''.join(header))
            if not isinstance(info, dict) or set(info) != {'name', 'description'}:
                raise ValueError('本课加载器仅支持 name 和 description 两字段')
            name, desc = info['name'], info['description']
            if (not isinstance(name, str) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*', name)
                    or len(name) > 64 or name != path.parent.name):
                raise ValueError('Skill 名称与目录不匹配或不合法')
            if not isinstance(desc, str) or not 1 <= len(desc.strip()) <= 1024:
                raise ValueError('description 必须为 1–1024 字符')
            self.entries[name] = info
        self.events.append({'stage': 'discover', 'names': list(self.entries)})

    def catalog(self):
        return [dict(x) for x in self.entries.values()]

    def activate(self, name):
        if name not in self.entries: raise ValueError('Skill 未注册')
        text = (self.root / name / 'SKILL.md').read_text(encoding='utf-8')
        self.active.add(name)
        self.events.append({'stage': 'activate', 'name': name})
        return text

    def read_resource(self, name, path):
        if name not in self.active: raise PermissionError('需要先激活 Skill')
        # 这是本课显式可读清单，不是所有 Skill 的通用路径规则。
        if path != 'references/conditions.md': raise PermissionError('资源未在本课清单中')
        target = self.root / name / path
        if target.is_symlink() or target.parent.is_symlink(): raise PermissionError('不读取符号链接')
        if target.stat().st_size > 16000: raise ValueError('资源过大')
        self.events.append({'stage': 'resource', 'name': name, 'path': path})
        return target.read_text(encoding='utf-8')


## 4. 激活正文后，再按任务读取参考文件

我们先显式选择 `repo-guide`，模拟宿主已经决定该 Skill 适用于当前任务。此处选择由代码指定，下一份 Notebook 再观察 DeepSeek 自己调用加载工具。

当前问题需要解释路径与缺失文件条件，因此正文中的“条件核验清单”相关；如果只是列出文件，未必需要读取这份清单。


In [5]:
instructions = library.activate("repo-guide")
print(instructions)
print("当前阶段:", library.events[-1])
reference = library.read_resource("repo-guide", "references/conditions.md")
print(reference)
print("当前阶段:", library.events[-1])


---
name: repo-guide
description: 带初学者理解小型代码仓库的入口、调用关系和条件分支。用于仓库导读、寻找配置入口、解释源码；每项结论回到实际读取的文件行号。
---
# 仓库导读

先用资料工具查看可读文件，再围绕用户问题读取入口及被调用函数。资料可能不完整；未读到实现时说明缺口，不根据文件名猜测行为。

解释配置、路径、默认值或异常分支时，按需读取 [条件核验清单](references/conditions.md)。该清单提供核验方法，不预先提供任何仓库的答案。

回答先给一条简短调用关系，再把关键条件讲清楚，引用实际文件和行号。给初学者留一个可以回到源码验证的问题。只回答用户所问范围，不为了使用 Skill 扩大任务。

Skill 正文中的步骤不授予运行程序或修改文件的权限。工具可执行范围由运行时和服务端控制。

当前阶段: {'stage': 'activate', 'name': 'repo-guide'}
# 条件核验清单

- 区分变量未设置、变量为空、文件不存在、文件内容非法；只解释源码实际覆盖的分支。
- 遇到相对路径，寻找是否显式拼接目录或改变进程工作目录；证据不足时说明已知的路径表达式和待确认的运行条件。
- 默认值只属于触发它的条件，不要当成所有配置的固定值。
- 读取 JSON 不自动意味着过滤字段。检查是否有显式键选择、校验或合并逻辑。
- 静态读源码与实际运行分开表述。没有执行记录时，用“按这段代码”而不是“已经运行证明”。

当前阶段: {'stage': 'resource', 'name': 'repo-guide', 'path': 'references/conditions.md'}


### 三个阶段分别给模型增加了什么

配套课件：L05.05-S01、L05.05-S02。

用全新的 SkillLibrary 重做发现、激活、读取资源，避免之前已激活的状态影响观察。比较的是每次返回文本的字符数，不是 Token 计费，也不能据此直接断言效果更好。

In [6]:
case_library = SkillLibrary(CHAPTER / "代码/skills")
case_catalog = case_library.catalog()
print("发现:", json.dumps(case_catalog, ensure_ascii=False))
assert not case_library.active
case_body = case_library.activate("repo-guide")
case_reference = case_library.read_resource("repo-guide", "references/conditions.md")
print("Skill 正文字符:", len(case_body), "参考文件字符:", len(case_reference))
print("事件顺序:", case_library.events)
assert "repo-guide" in case_library.active
assert case_body and case_reference and case_body != case_reference


发现: [{"name": "repo-guide", "description": "带初学者理解小型代码仓库的入口、调用关系和条件分支。用于仓库导读、寻找配置入口、解释源码；每项结论回到实际读取的文件行号。"}]
Skill 正文字符: 382 参考文件字符: 223
事件顺序: [{'stage': 'discover', 'names': ['repo-guide']}, {'stage': 'activate', 'name': 'repo-guide'}, {'stage': 'resource', 'name': 'repo-guide', 'path': 'references/conditions.md'}]


**结果解读**

发现阶段返回名称和描述，激活才返回 Skill 正文，读取资源再增加条件检查细节。文件在包里不等于本次模型上下文已包含它；调用方还需把返回值交给模型。

**小练习**

若先读取 references/conditions.md，再 activate，当前课程加载器会怎样？它是在限制哪些动作？

<details><summary>完成后展开参考分析</summary>

会拒绝未激活的资源读取。它限制该加载器的资源访问流程，不授予任意文件系统、网络或执行权限。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 5. 怎样写出有用的导读 Skill

第一句说明使用场景；正文改变具体阅读行为，例如先找入口、再查调用方与被调用方；遇到条件分支才读清单。输出要求引用文件与行号，让学生能够回查。

不要在 Skill 正文里提前写死某个仓库的答案，也不要承诺“用了 Skill 就一定准确”。Skill 是否有用必须接受实际任务检验。

练习只改描述，使“读源码导读”和“纯聊天问候”的使用范围更容易区分。把自己的版本写入课程练习目录，不覆盖随课原始文件。


In [7]:
import tempfile
student_description = "帮助初学者从入口追踪代码仓库中的调用与配置分支，用于源码导读和定位实现；要求结论引用已读取的文件行号。"
with tempfile.TemporaryDirectory() as temp_dir:
    package = Path(temp_dir) / "repo-guide"
    package.mkdir()
    text = skill_path.read_text(encoding='utf-8')
    # 用 YAML 生成合法头部，保留随课 Skill 正文。
    body = text.split("---", 2)[2]
    header = yaml.safe_dump({"name":"repo-guide", "description":student_description}, allow_unicode=True)
    (package / "SKILL.md").write_text("---\n" + header + "---\n" + body, encoding='utf-8')
    student_library = SkillLibrary(temp_dir)
    print(student_library.catalog())


[{'description': '帮助初学者从入口追踪代码仓库中的调用与配置分支，用于源码导读和定位实现；要求结论引用已读取的文件行号。', 'name': 'repo-guide'}]


In [8]:
assert "源码导读" in student_description
print("格式检查通过。请再人工判断两个任务：解释配置入口；和我打个招呼。")
print("前者与描述匹配；后者无需为使用 Skill 而读取仓库。关键词检查不等于完整的适用性评估。")


格式检查通过。请再人工判断两个任务：解释配置入口；和我打个招呼。
前者与描述匹配；后者无需为使用 Skill 而读取仓库。关键词检查不等于完整的适用性评估。


## 6. Skill 指引与执行权限分开

试着在未激活时读取参考，再试着读取清单外路径。观察是模型“答应了就能读”，还是运行时实际拒绝。这里的规则由 Python 实现，正文只是说明。


In [9]:
fresh = SkillLibrary(CHAPTER / "代码/skills")
for name, path in [("repo-guide", "references/conditions.md")]:
    try: fresh.read_resource(name, path)
    except PermissionError as error: print("预期拒绝:", error)
fresh.activate("repo-guide")
try: fresh.read_resource("repo-guide", "../outside.txt")
except PermissionError as error: print("预期拒绝:", error)


预期拒绝: 需要先激活 Skill
预期拒绝: 资源未在本课清单中


## 7. 交付：Skill 加载审计

最后不只展示文件内容，而是记录这个包实际包含什么、发现阶段看到什么、激活和按需读取是否分开，以及运行时边界是否真的拒绝了不合法读取。

格式通过不等于 Skill 对任务有效；报告会把“学习者还需要人工判断的改进点”单独列出。


In [10]:
RUN_DIR = ROOT / ".local/student-runs/l05"
RUN_DIR.mkdir(parents=True, exist_ok=True)
package_files = sorted(str(p.relative_to(skill_path.parent))
                       for p in skill_path.parent.rglob("*") if p.is_file())
boundary_checks = []
check_library = SkillLibrary(CHAPTER / "代码/skills")
try:
    check_library.read_resource("repo-guide", "references/conditions.md")
except PermissionError as error:
    boundary_checks.append({"case": "resource_before_activation", "status": "denied", "error": str(error)})
check_library.activate("repo-guide")
try:
    check_library.read_resource("repo-guide", "../outside.txt")
except PermissionError as error:
    boundary_checks.append({"case": "resource_outside_allowlist", "status": "denied", "error": str(error)})

skill_audit = {
    "artifact": "L05-skill-package-audit",
    "package": library.catalog()[0],
    "files": package_files,
    "progressive_loading_events": library.events,
    "boundary_checks": boundary_checks,
    "learner_revision": {
        "description": student_description,
        "reason": "把源码导读、实现定位和行号证据写入适用范围",
        "still_requires_review": ["false_positive_activation", "method_quality", "host_permission_enforcement"],
    },
}
skill_audit_path = RUN_DIR / "skill-package-audit.json"
skill_audit_path.write_text(json.dumps(skill_audit, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
assert [x["status"] for x in boundary_checks] == ["denied", "denied"]
print("已生成 Skill 加载审计:", skill_audit_path.relative_to(ROOT))
print("加载阶段:", [x["stage"] for x in library.events])


已生成 Skill 加载审计: .local\student-runs\l05\skill-package-audit.json
加载阶段: ['discover', 'activate', 'resource']


## 回顾

名称与描述让应用知道有哪些 Skills；激活后读取 `SKILL.md` 正文，任务需要时再读参考。发现 Skill、选择 Skill、读取文件、送入模型是可以分别观察的动作。

本节生成了 `skill-package-audit.json`，把包元数据、实际文件、渐进加载事件、边界拒绝和学习者修订留在同一份记录中。

下一份实验把它与真实 MCP 工具接在一起，核验过程与最终结果。

来源：[Agent Skills 规范](https://agentskills.io/specification)。
